In [1]:
# ESTRUTURAÇÃO E SINCRONIZAÇÃO DINÂMICA

import os
import glob
import pandas as pd
import numpy as np

print("Notebook 1: Estruturação e sincronização dinâmica")
print("="*80)

# 1. Definição de Caminhos
PASTA_RAW = '/workspaces/EyeTracking/data/raw/'
ARQUIVO_TRANSICOES = '/workspaces/EyeTracking/data/transicoes/transicoes_estados.csv'
PASTA_CSV = '/workspaces/EyeTracking/data/csv/'

os.makedirs(PASTA_CSV, exist_ok=True)

# 2. Carregar e Estruturar o Gabarito de Transições
try:
    df_trans = pd.read_csv(ARQUIVO_TRANSICOES)
    # Criar a coluna de tempo em milissegundos
    df_trans['tempo_ms'] = df_trans['segundos'] * 1000.0
    
    # Cria os limites de tempo (bins) para a busca rápida
    # Adiciona 0 no início e infinito no final para cobrir todo o arquivo
    bins = [0.0] + list(df_trans['tempo_ms']) + [float('inf')]
    
    # Prepara os rótulos correspondentes a cada intervalo
    estados = ['BRANCO'] + list(df_trans['estado_novo'])
    
    # Para o Face_ID, a transição 'CRUZ_FIX' no CSV já aponta para o 'n_estado_novo' da próxima face.
    n_faces = [0] + list(df_trans['n_estado_novo'])
    
    print(f" Gabarito de transições carregado. Total de eventos mapeados: {len(df_trans)}")
except Exception as e:
    print(f" Erro ao ler o arquivo de transições: {e}")
    raise

# 3. Processamento dinâmico dos dados brutos
arquivos_raw = glob.glob(os.path.join(PASTA_RAW, '*.xlsx'))
EXCLUSOES_DEFINITIVAS = ['T12', 'T15', 'T42']

arquivos_validos = []
for caminho in arquivos_raw:
    nome_puro = os.path.basename(caminho).replace('.xlsx', '')
    if any(excluido in nome_puro for excluido in EXCLUSOES_DEFINITIVAS):
        print(f" Bloqueado: '{nome_puro}' retido (Arquivo corrompido/Aguardando recuperação).")
    else:
        arquivos_validos.append(caminho)

print(f"\nIniciando sincronização de {len(arquivos_validos)} arquivos válidos...")

contador_sucesso = 0

# O loop itera sobre 'arquivos_validos' e não sobre os raw originais
for caminho in arquivos_validos:
    nome_puro = os.path.basename(caminho).replace('.xlsx', '')
    
    try:
        # Lê os dados originais brutos
        df = pd.read_excel(caminho)
        
        # Garante que o tempo é numérico
        df['Time (ms)'] = pd.to_numeric(df['Time (ms)'], errors='coerce')
        df = df.dropna(subset=['Time (ms)']).copy()
        
        # A. Sincronização vetorial
        # pd.cut descobre em qual 'bin' (intervalo) o milissegundo se encontra
        bin_indices = pd.cut(df['Time (ms)'], bins=bins, labels=False, right=False)
        
        # Injeta as informações do gabarito direto na linha de dados oculares
        df['Estado_Bruto'] = np.array(estados)[bin_indices]
        df['Face_ID'] = np.array(n_faces)[bin_indices].astype(int)
        
        # Captura o tempo de início e fim daquele estado exato
        df['Inicio_Estado_ms'] = np.array(bins[:-1])[bin_indices]
        df['Fim_Estado_ms'] = np.array(bins[1:])[bin_indices]
        
        # B. Classificação de estímulos e emoções
        df['Tipo_Estimulo'] = np.where(df['Face_ID'] == 0, 'Nenhum',
                              np.where(df['Face_ID'] <= 30, 'Humano', 'Desenho'))
        
        condicoes_emocao = [
            (df['Face_ID'] == 0),
            # Humanos (1 a 30): Feliz(1), Neutro(2), Raiva(0)
            (df['Tipo_Estimulo'] == 'Humano') & (df['Face_ID'] % 3 == 1),
            (df['Tipo_Estimulo'] == 'Humano') & (df['Face_ID'] % 3 == 2),
            (df['Tipo_Estimulo'] == 'Humano') & (df['Face_ID'] % 3 == 0),
            # Desenhos (31 a 50): Neutro(1), Raiva(2), Feliz(0)
            (df['Tipo_Estimulo'] == 'Desenho') & (df['Face_ID'] % 3 == 1),
            (df['Tipo_Estimulo'] == 'Desenho') & (df['Face_ID'] % 3 == 2),
            (df['Tipo_Estimulo'] == 'Desenho') & (df['Face_ID'] % 3 == 0)
        ]
        opcoes_emocao = [
            'Nenhuma',
            'Feliz', 'Neutro', 'Raiva',
            'Neutro', 'Raiva', 'Feliz'
        ]
        df['Emocao'] = np.select(condicoes_emocao, opcoes_emocao, default='Desconhecido')
        
        # C. Regras de marcação temporal (Sem descartes)
        df['Fase_Estimulo'] = 'Descarte' # Começa assumindo que é descartável (ex: intervalo em branco)
        
        # A janela de cruz fixa é inteira aproveitada como baseline
        df.loc[df['Estado_Bruto'] == 'CRUZ_FIX', 'Fase_Estimulo'] = 'Janela_Baseline'
        
        # A Face é aproveitada integralmente (0ms até o final) para capturar o TTFF real
        mask_face_segura = (df['Estado_Bruto'] == 'FACE')
        df.loc[mask_face_segura, 'Fase_Estimulo'] = 'Exposicao_Face'
        
        # D. Higienização estrutural final
        # Padroniza nomes e isola variáveis cruciais
        colunas_finais = {
            'Time (ms)': 'Time_ms',
            'Gaze X': 'Gaze_X',
            'Gaze Y': 'Gaze_Y',
            'Pupil Diameter Left (mm)': 'Pupil_L',
            'Pupil Diameter Right (mm)': 'Pupil_R',
            'Face_ID': 'Face_ID',
            'Tipo_Estimulo': 'Tipo_Estimulo',
            'Emocao': 'Emocao',
            'Fase_Estimulo': 'Fase_Estimulo'
        }
        
        # Manter apenas as colunas que existem no raw original (caso falte alguma pupila, o código não quebra)
        colunas_presentes = {k: v for k, v in colunas_finais.items() if k in df.columns or k in ['Face_ID', 'Tipo_Estimulo', 'Emocao', 'Fase_Estimulo']}
        
        df_limpo = df[list(colunas_presentes.keys())].rename(columns=colunas_presentes)
        
        # Salvar o CSV final pronto para o ML
        caminho_salvar = os.path.join(PASTA_CSV, f"{nome_puro}.csv")
        df_limpo.to_csv(caminho_salvar, index=False)
        contador_sucesso += 1
        
    except Exception as e:
        print(f" Erro ao processar '{nome_puro}': {str(e)}")

print("\n" + "="*80)
print(f"SINCRONIZAÇÃO CONCLUÍDA: {contador_sucesso} arquivos salvos.")
print("Verifique a pasta data/csv/. Os dados estão ancorados ao arquivo de transições e preservando 100% da exposição visual.")

Notebook 1: Estruturação e sincronização dinâmica
 Gabarito de transições carregado. Total de eventos mapeados: 94
 Bloqueado: 'T12 CONTROLE' retido (Arquivo corrompido/Aguardando recuperação).
 Bloqueado: 'T42 CONTROLE' retido (Arquivo corrompido/Aguardando recuperação).
 Bloqueado: 'T15 TEA MODERADO' retido (Arquivo corrompido/Aguardando recuperação).

Iniciando sincronização de 38 arquivos válidos...

SINCRONIZAÇÃO CONCLUÍDA: 38 arquivos salvos.
Verifique a pasta data/csv/. Os dados estão ancorados ao arquivo de transições e preservando 100% da exposição visual.
